# Домашнее задание 3

## Задание 1 (синтаксис)

Скачайте датасет Kaggle electronic_devices_sales и выполните задания ниже.

Для каждого покупателя получите таблицу со столбцами:
- preferred_payment - самый частый метод оплаты (при ничьей зафиксируйте правило, например lexicographically первый);
- total_spent - сумма всех трат;
- addons_spent - сумма на дополнительные услуги и аксессуары;
- orders_count - число заказов.
 
 Отсортируйте по total_spent по убыванию и выведите топ-10.

Пример вывода (формат; числа зависят от датасета):

||customer_id|preferred_payment|total_spent|addons_spent|orders_count|
|-|-|-|-|-|-|
|0|C042|Credit Card|15420.50|890.00|7|
|1|C018|PayPal|14810.00|450.00|5|
|2|C103|Debit Card|13990.25|1200.00|6|
 ...

Пограничные случаи:

- Покупатель с одним заказом - orders_count = 1, preferred_payment = единственный метод.
- Два метода оплаты с одинаковой частотой - выберите один по зафиксированному правилу (укажите его в комментарии к коду).
- Покупатель без доп. услуг/аксессуаров - addons_spent = 0.
- Пропуски в сумме или методе оплаты - обработайте (отфильтруйте или заполните) и кратко опишите решение в комментарии.

In [1]:
import pandas as pd

In [2]:
esales = pd.read_csv('Electronic_sales_Sep2023-Sep2024.csv')
esales.head()

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,1000,53,Male,No,Smartphone,SKU1004,2,Cancelled,Credit Card,5538.33,791.19,7,2024-03-20,Standard,"Accessory,Accessory,Accessory",40.21
1,1000,53,Male,No,Tablet,SKU1002,3,Completed,Paypal,741.09,247.03,3,2024-04-20,Overnight,Impulse Item,26.09
2,1002,41,Male,No,Laptop,SKU1005,3,Completed,Credit Card,1855.84,463.96,4,2023-10-17,Express,NaN,0.00
3,1002,41,Male,Yes,Smartphone,SKU1004,2,Completed,Cash,3164.76,791.19,4,2024-08-09,Overnight,"Impulse Item,Impulse Item",60.16
4,1003,75,Male,Yes,Smartphone,SKU1001,5,Completed,Cash,41.50,20.75,2,2024-05-21,Express,Accessory,35.56


In [3]:
esales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Customer ID        20000 non-null  int64  
 1   Age                20000 non-null  int64  
 2   Gender             19999 non-null  object 
 3   Loyalty Member     20000 non-null  object 
 4   Product Type       20000 non-null  object 
 5   SKU                20000 non-null  object 
 6   Rating             20000 non-null  int64  
 7   Order Status       20000 non-null  object 
 8   Payment Method     20000 non-null  object 
 9   Total Price        20000 non-null  float64
 10  Unit Price         20000 non-null  float64
 11  Quantity           20000 non-null  int64  
 12  Purchase Date      20000 non-null  object 
 13  Shipping Type      20000 non-null  object 
 14  Add-ons Purchased  15132 non-null  object 
 15  Add-on Total       20000 non-null  float64
dtypes: float64(3), int64(4

In [4]:
esales.shape

(20000, 16)

In [5]:
# Группировка и расчет метрик
result = esales.groupby('Customer ID').agg(
    preferred_payment=('Payment Method', lambda x: x.mode().min()), # При ничьей — первый по алфавиту
    total_spent=('Total Price', 'sum'),                            # Сумма всех трат
    addons_spent=('Add-on Total', 'sum'),                          # Сумма трат на доп. услуги
    orders_count=('Customer ID', 'count')                          # Количество заказов
).reset_index()

In [6]:
# Приведение названий колонок к требуемому формату
result = result.rename(columns={'Customer ID': 'customer_id'})

In [7]:
# Сортировка по total_spent по убыванию и выбор ТОП-10
top10_customers = result.sort_values(by='total_spent', ascending=False).head(10)

top10_customers

,customer_id,preferred_payment,total_spent,addons_spent,orders_count
9728,16357,Bank Transfer,34563.70,495.19,7
10049,16863,PayPal,33035.92,486.17,5
8060,13813,Credit Card,31830.16,268.64,5
6475,11476,PayPal,31077.61,301.59,5
7028,12276,Bank Transfer,30961.18,329.07,6
7935,13635,Credit Card,30260.36,480.73,5
7352,12749,Credit Card,29394.56,619.43,5
9087,15399,PayPal,29084.88,305.39,3
7054,12319,Bank Transfer,27352.32,226.38,3
12133,19996,Bank Transfer,27296.78,432.12,6
